# Exercise 12.3: Computing for a future collider

From *Programming in High Energy Particle Physics*, Chapter 12

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch12/ex12_3_solution.ipynb)

Exercise 12.3 Computing for a future collider Research one of the proposed future colliders (FCC-hh, CLIC, ILC, or muon collider). Estimate the annual data volume and computing requirements, and compare with the HL-LHC. What new software challenges would such a facility face?

<details><summary>Hint</summary>

Start from the design reports (FCC Conceptual/Feasibility Study reports, CLIC and ILC Technical Design Reports, the International Muon Collider Collaboration documents). Estimate data volume as event rate after trigger × event size × live time (about \(10^7\) s per year). For FCC-hh, pileup of about 1000 at 100 TeV dominates the problem; for \(e^+e^-\) machines the rates are low but precision demands on simulation and calibration are extreme; for a muon collider, beam-induced background from muon decays is the main challenge.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib")

**Sources and assumptions:** FCC-hh conceptual detector study (CDS record 2842569) supports the 100 TeV, pileup ≈1000 conceptual benchmark. CMS Phase-2 DAQ/HLT TDR (CDS record 2759072) gives 7.5 kHz output. [FCC-hh feasibility-era note](https://repository.cern/records/mf0cz-2c516/files/FCC_PED_ESPPU_FCC_hh.pdf?download=1) describes an 84 TeV baseline. Event sizes, FCC accepted rate and CPU seconds per event below are deliberately stated assumptions to vary, not sourced design numbers.

### Step 1: Set explicit FCC-hh and HL-LHC assumptions

In [ ]:
import numpy as np
LIVE_SECONDS=10_000_000
PROCESSING_SECONDS=365*24*3600  # full calendar year, ideal 100% utilization
# Explicit illustrative scenarios, not approved experiment design parameters.
scenarios={'HL-LHC CMS example':{'rate_hz':7500,'size_mb':1.0,'cpu_s_per_event':3.0},
           'FCC-hh conceptual example':{'rate_hz':10000,'size_mb':5.0,'cpu_s_per_event':10.0}}
assert LIVE_SECONDS>0

### Step 2: Estimate annual storage and CPU capacity

In [ ]:
for name,c in scenarios.items():
    events=c['rate_hz']*LIVE_SECONDS
    pb=events*c['size_mb']*1e6/1e15
    cores=events*c['cpu_s_per_event']/PROCESSING_SECONDS
    print(f'{name}: {events/1e9:.0f} billion events/year, {pb:.0f} PB/year, {cores:,.0f} ideal year-round busy cores')
ratio=(scenarios['FCC-hh conceptual example']['rate_hz']*scenarios['FCC-hh conceptual example']['size_mb'])/(scenarios['HL-LHC CMS example']['rate_hz']*scenarios['HL-LHC CMS example']['size_mb'])
print('FCC/HL-LHC illustrative volume ratio:',round(ratio,2))

### Step 3: Explore sensitivity and list software challenges

In [ ]:
import matplotlib.pyplot as plt
rates=np.array([5,10,20])*1000; sizes=np.array([2,5,10]); volumes=np.outer(rates,sizes)*LIVE_SECONDS*1e6/1e15
fig,ax=plt.subplots(figsize=(7,4)); im=ax.imshow(volumes,origin='lower',aspect='auto')
ax.set(xticks=np.arange(len(sizes)),xticklabels=sizes,yticks=np.arange(len(rates)),yticklabels=rates//1000,
       xlabel='Stored event size [MB]',ylabel='Accepted rate [kHz]')
fig.colorbar(im,ax=ax,label='Annual data volume [PB/year]'); plt.show()

The example assumptions give 500 PB/year for FCC-hh versus 75 PB/year for the HL-LHC CMS comparison, a 6.7× ratio. These are scenario outputs, not design predictions. The CPU estimates assume full-year processing at 100% utilization; real facilities need more capacity. The FCC-hh conceptual detector study discusses about 1000 interactions per crossing at 100 TeV; the newer feasibility baseline considers about 84 TeV. Major software challenges include dense pileup reconstruction, fast simulation, trigger inference, calibration, storage, data movement and energy efficiency.